[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-pca.ipynb)

# Principal Component Analysis

*AIBits Academy · Machine Learning End To End · Dimensionality Reduction*

PCA finds the orthogonal axes of maximum variance in high-dimensional data — compressing features while preserving the most information.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Motivation

High-dimensional datasets suffer from: the curse of dimensionality (distances concentrate), visualisation difficulty, and redundant correlated features. PCA transforms the original p features into k uncorrelated **principal components** (k ≪ p) that capture the most variance.

## The Math

$$\begin{gathered}1.\ \text{Centre data: } X_c = X - \mathrm{mean}(X) \\[6pt] 2.\ \text{Covariance matrix: } \Sigma = \tfrac{1}{n} X_c^\mathsf{T} X_c \\[6pt] 3.\ \text{Eigendecomposition: } \Sigma = V\Lambda V^\mathsf{T} \ \text{(sort by eigenvalue descending)} \\[6pt] 4.\ \text{Project: } Z = X_c \cdot V_k \ \text{(top } k \text{ eigenvectors)}\end{gathered}$$

> **📊 Prerequisite refresher**
>
> Step 3 is the crux of PCA and leans entirely on eigenvalues/eigenvectors from linear algebra — see the **Linear Algebra for ML** prerequisite page for the full derivation from matrices and vectors up through eigendecomposition, including this exact covariance-matrix example worked out with real numbers. Core intuition: eigenvectors of the covariance matrix are the directions of maximum spread, and their eigenvalues tell you exactly how much variance lies along each one.

> **⚙ Deriving Why Eigenvectors — The Full Argument**
>
> Step 3 isn't just asserted — it falls directly out of solving PCA's actual objective. The first principal component is the unit vector **w** (‖w‖=1) that maximises the variance of the data projected onto it:
>
> $$\mathrm{Var}(X_c w) = \tfrac{1}{n}(X_c w)^\mathsf{T}(X_c w) = w^\mathsf{T}\Sigma w \qquad \left(\text{since } \Sigma = \tfrac{1}{n}X_c^\mathsf{T}X_c\right)$$
>
> Maximise wᵀΣw subject to wᵀw = 1 using a Lagrange multiplier:
>
> $$L(w,\lambda) = w^\mathsf{T}\Sigma w - \lambda(w^\mathsf{T}w - 1) \;\;\to\;\; \frac{\partial L}{\partial w} = 2\Sigma w - 2\lambda w = 0 \;\;\to\;\; \Sigma w = \lambda w$$
>
> The critical points of the variance are exactly the eigenvectors of Σ — that's the whole reason eigendecomposition appears in Step 3. Plugging Σw=λw back into the objective gives wᵀΣw = wᵀ(λw) = λ(wᵀw) = λ, so **the variance captured along direction w equals its eigenvalue**. The maximum over all unit vectors is therefore the *largest* eigenvalue λ₁, achieved at its eigenvector v₁ — PC1. Each subsequent PC repeats this maximisation subject to being orthogonal to all previous PCs, which is exactly why the remaining eigenvectors (in descending eigenvalue order) are PC2, PC3, and so on.

> **🔁 Two Equivalent Views of PCA**
>
> Everything above frames PCA as **maximising variance**. There's a second, completely equivalent way to arrive at the same eigenvectors: **minimising reconstruction error** — finding the k-dimensional subspace that, when you project the data onto it and reconstruct back to the original space, loses the least information (smallest mean squared error). The Eckart–Young theorem guarantees these two objectives have identical solutions: the top-k eigenvectors of Σ. This is also exactly what the SVD of X_c computes directly, which is why scikit-learn's `PCA` is implemented via SVD internally rather than by literally eigendecomposing the covariance matrix.

## PCA Axes Visualisation

## From Scratch with NumPy

In [ ]:
import numpy as np

np.random.seed(42)
n = 200
X = np.column_stack([
    np.random.normal(5, 3, n),
    np.random.normal(12, 5, n) + np.random.normal(5,3,n)*1.8,
    np.random.normal(3.5, 0.8, n),
    np.random.normal(8, 4, n),
    np.random.normal(4, 2, n),
])
X_c = X - X.mean(axis=0)
cov = X_c.T @ X_c / (n - 1)
eigenvalues, eigenvectors = np.linalg.eigh(cov)
idx = np.argsort(-eigenvalues)
eigenvalues = eigenvalues[idx]; eigenvectors = eigenvectors[:, idx]
exp_var = eigenvalues / eigenvalues.sum()
print("Explained variance ratio:")
for i,(e,v) in enumerate(zip(exp_var, exp_var.cumsum())):
    print(f"  PC{i+1}: {e:.3f}  Cumulative: {v:.3f}")
X_pca = X_c @ eigenvectors[:, :2]
print(f"\nOriginal: {X.shape} → Reduced: {X_pca.shape}")
print(f"Variance retained: {exp_var[:2].sum():.3f}")

## With scikit-learn — Scree Plot Strategy

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_s = StandardScaler().fit_transform(X)
pca_full = PCA()
pca_full.fit(X_s)
# Choose k where cumulative variance ≥ 95%
cumvar = pca_full.explained_variance_ratio_.cumsum()
k95 = (cumvar >= 0.95).argmax() + 1
print(f"Components needed for 95% variance: {k95}")

# Rebuild the pipeline with PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

y_dummy = (X[:,0] > 5).astype(int)
pipe = Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=k95)),
                 ('clf', LogisticRegression())])
cv = cross_val_score(pipe, X, y_dummy, cv=5)
print(f"CV accuracy with PCA preprocessing: {cv.mean():.3f}")

## Supervised Alternative: Linear & Quadratic Discriminant Analysis

PCA is **unsupervised** dimensionality reduction — it finds directions of maximum variance with no knowledge of class labels, which can occasionally discard a low-variance direction that actually separates classes perfectly. **Linear Discriminant Analysis (LDA)** instead uses the labels directly, finding the projection that *maximises class separability*:

$$\text{maximise } J(w) = \dfrac{w^\mathsf{T} S_B w}{w^\mathsf{T} S_W w} \qquad S_B = \text{between-class scatter},\ S_W = \text{within-class scatter}$$

This ratio is maximised when classes are pushed far apart (large S_B) while each class stays tightly clustered internally (small S_W) — precisely the notion of "good separation" a classifier needs. LDA assumes all classes share the same covariance structure, giving it a **linear** decision boundary. **Quadratic Discriminant Analysis (QDA)** relaxes that assumption, letting each class have its own covariance matrix — at the cost of more parameters to estimate, needing more data per class to avoid overfitting.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.model_selection import cross_val_score

# HDFC loan approval — same feature set as the Decision Trees chapter
lda = LinearDiscriminantAnalysis()
qda = QuadraticDiscriminantAnalysis()
for name, model in [('LDA',lda), ('QDA',qda), ('PCA+LogReg',pipe)]:
    scores = cross_val_score(model, X, y_dummy, cv=5)
    print(f"{name:12s}  CV accuracy: {scores.mean():.3f}")
# LDA can ALSO be used purely for dimensionality reduction, like PCA —
# but projects onto at most (n_classes - 1) axes that maximise separability

LDA edges out PCA+LogisticRegression here precisely because it optimises directly for separability rather than variance — but this comes at a cost: LDA needs the labels, so it can never be used in a purely unsupervised setting (e.g., exploratory visualisation of unlabelled data), which is exactly where PCA remains the right tool.

## ⚠ Advanced: Factor Analysis

PCA finds directions of maximum variance with no underlying statistical model of *why* the data looks the way it does. **Factor Analysis** instead assumes each observed feature is a linear combination of a small number of unobserved (latent) factors plus feature-specific noise:

$$\mathbf{x} = \Lambda \mathbf{f} + \boldsymbol{\varepsilon} \qquad f = \text{latent factors},\ \Lambda = \text{factor loadings},\ \varepsilon = \text{feature-specific noise (NOT shared across features)}$$

This is a subtle but important distinction from PCA: Factor Analysis explicitly separates "shared signal" (the factors) from "noise unique to each feature," while PCA's components can absorb both indiscriminately. This makes Factor Analysis the traditional tool of choice in psychometrics and social science (e.g., inferring a handful of latent "customer satisfaction dimensions" from dozens of survey questions, each with its own measurement noise).

In [ ]:
from sklearn.decomposition import FactorAnalysis, PCA

# 12 survey questions assumed to reflect 3 underlying satisfaction dimensions
fa = FactorAnalysis(n_components=3, random_state=42).fit(X)
# noise_variance_ is Factor Analysis' key output PCA has no equivalent for —
# per-feature noise NOT explained by the shared latent factors
print(f"Per-feature noise variance: {np.round(fa.noise_variance_, 3)}")

## ⚠ Advanced: Independent Component Analysis (ICA)

PCA finds *uncorrelated* components (zero covariance). ICA searches for a stronger condition — **statistically independent** components — making it the standard tool for **blind source separation**: recovering original independent signals that have been linearly mixed together, without knowing the mixing process in advance.

$$\mathbf{x} = A\mathbf{s} \qquad s = \text{independent source signals (unknown)},\ A = \text{mixing matrix (unknown)} \qquad \text{ICA recovers } \hat{\mathbf{s}} = W\mathbf{x}$$

The classic illustration is the "cocktail party problem": several microphones each record a mixture of multiple people talking simultaneously; ICA can separate the mixed recordings back into each individual speaker's voice, using only statistical independence — no information about microphone placement or voice characteristics required. Since this is a purely mathematical/synthetic illustration of the mixing process itself, the example below uses three genuinely independent synthetic signals rather than a forced business framing.

In [ ]:
import numpy as np
from sklearn.decomposition import FastICA, PCA

# Three genuinely independent, non-Gaussian source signals
time = np.linspace(0, 8, 2000)
s1 = np.sin(2 * time)                    # sine wave
s2 = np.sign(np.sin(3 * time))           # square wave
s3 = np.random.RandomState(42).laplace(size=2000)  # noisy Laplace source
S = np.c_[s1, s2, s3]
S /= S.std(axis=0)

# Mix the three sources together with an arbitrary 3x3 mixing matrix
A = np.array([[1,1,1], [0.5,2,1.0], [1.5,1.0,2.0]])
X_mixed = S.dot(A.T)  # only X_mixed would be "observed" in a real cocktail-party recording

ica = FastICA(n_components=3, random_state=42, whiten='unit-variance').fit(X_mixed)
S_ica = ica.transform(X_mixed)
pca = PCA(n_components=3, random_state=42).fit(X_mixed)
S_pca = pca.transform(X_mixed)

# Best-match correlation between each TRUE source and its recovered version
corr_ica = np.abs(np.corrcoef(S.T, S_ica.T)[:3, 3:]).max(axis=1)
corr_pca = np.abs(np.corrcoef(S.T, S_pca.T)[:3, 3:]).max(axis=1)
print(f"ICA mean recovery correlation:  {corr_ica.mean():.4f}")
print(f"PCA mean recovery correlation:  {corr_pca.mean():.4f}")

ICA recovers the three original, independently-generated signals almost perfectly (mean correlation 0.9991) from nothing but the mixed observations — it never sees `S`, `A`, `s1`, `s2`, or `s3` directly. Plain PCA, run on the exact same mixed input, manages only 0.7280: PCA can find the directions of maximum variance in the mixture, but decorrelating the mixed signals is not the same as recovering statistically *independent* ones — a sine wave and a square wave can easily be made uncorrelated (zero covariance) while remaining trivially predictable from one another, which is precisely the kind of residual dependency PCA cannot detect but ICA is built to exploit.

> **⚠ PCA vs ICA — Not Interchangeable**
>
> PCA answers "what directions capture the most variance?" — useful for compression and denoising. ICA answers "what original independent signals were mixed together?" — useful for un-mixing. Using PCA when you actually need source separation (or vice versa) produces components that are mathematically valid but meaningless for the actual question being asked. The 0.9991 vs. 0.7280 recovery gap above is the direct, measured consequence of that mismatch.

## ⚠ Advanced: Isomap & Multidimensional Scaling (MDS)

PCA, Factor Analysis and ICA all find a *linear* transformation of the original features. Some datasets, though, sit on a curved surface — the classic teaching example is the "Swiss roll," a flat 2D sheet rolled up into 3D space. A straight-line projection can't unroll it: two points genuinely far apart along the sheet can end up right next to each other in 3D purely because the roll happens to curl them close together. Since this is a purely mathematical illustration of manifold shape, the example below uses a synthetic dataset rather than a forced business framing.

**Isomap** and **Multidimensional Scaling (MDS)** are two classic non-linear ("manifold learning") alternatives built for exactly this situation:

| Method | Core idea | Best for |
|---|---|---|
| **MDS** | Directly preserves pairwise *straight-line* distances between all points as faithfully as possible in the low-dimensional space. | General-purpose distance-preserving layouts; simplest to explain. |
| **Isomap** | Preserves *geodesic* distance — distance measured *along the manifold's surface* via a neighbour graph — rather than straight-line distance through the ambient space. | Data known to lie on a curved surface, like the Swiss roll. |

The standard way to judge these embeddings is **trustworthiness**: for each point, what fraction of its nearest neighbours in the low-dimensional embedding were *also* its nearest neighbours in the original space? 1.0 means neighbourhoods are perfectly preserved.

In [ ]:
from sklearn.datasets import make_swiss_roll
from sklearn.manifold import Isomap, MDS, trustworthiness
from sklearn.decomposition import PCA

X, color = make_swiss_roll(n_samples=800, noise=0.05, random_state=42)

X_iso = Isomap(n_neighbors=10, n_components=2).fit_transform(X)
X_mds = MDS(n_components=2, random_state=42, normalized_stress='auto').fit_transform(X)
X_pca = PCA(n_components=2, random_state=42).fit_transform(X)

for name, X_low in [('Isomap', X_iso), ('MDS', X_mds), ('PCA', X_pca)]:
    print(f"{name:8s} trustworthiness: {trustworthiness(X, X_low, n_neighbors=10):.4f}")

On this Swiss roll, Isomap's geodesic-aware approach recovers the true neighbourhood structure almost perfectly (0.9992). Generic MDS (0.9240) and even plain linear PCA (0.9396) preserve straight-line distance/variance well enough to score respectably too — neither collapses — but neither one explicitly follows the manifold's curl the way Isomap does, which is precisely why Isomap exists as a distinct tool for data with known non-linear structure rather than a strict upgrade over MDS in every case.

> **🔗 Real-World Link — Instagram Recommendations**
>
> The same real Instagram engagement dataset used for reach analysis, viewed differently here: reducing 11 numeric engagement metrics to a handful of principal components to find posts with a similar underlying engagement "shape." [See the case study →](https://statso.io/instagram-recommendations-case-study/) ·

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · PCA from the covariance matrix

Centre `X`, compute its covariance matrix and the eigenvalues with `np.linalg.eigh`. Store the **explained-variance ratios** (largest first) in `ratios`; they must match scikit-learn's `PCA`.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
rng = np.random.default_rng(0)
z = rng.normal(size=(300, 1))
X = np.hstack([z * 3 + rng.normal(size=(300, 1)), z * -2 + rng.normal(size=(300, 1)), rng.normal(size=(300, 1))])
ratios = None   # TODO


In [ ]:
try:
    check("matches sklearn", np.allclose(ratios, PCA().fit(X).explained_variance_ratio_))
    check("sorted largest first", all(np.diff(ratios) <= 0))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.decomposition import PCA
rng = np.random.default_rng(0)
z = rng.normal(size=(300, 1))
X = np.hstack([z * 3 + rng.normal(size=(300, 1)), z * -2 + rng.normal(size=(300, 1)), rng.normal(size=(300, 1))])
vals = np.linalg.eigh(np.cov(X - X.mean(axis=0), rowvar=False))[0][::-1]
ratios = vals / vals.sum()

```

</details>

### Exercise 2 · Medium · How many components for 95%?

Standardise `X` and find the smallest number of principal components that explain at least 95% of the variance. Store it in `k95`.

In [ ]:
from sklearn.preprocessing import StandardScaler
k95 = None   # TODO (reuse X and PCA)


In [ ]:
try:
    check("two components are enough for this data", k95 == 2)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.preprocessing import StandardScaler
cum = PCA().fit(StandardScaler().fit_transform(X)).explained_variance_ratio_.cumsum()
k95 = int((cum >= 0.95).argmax()) + 1

```

</details>

### Exercise 3 · Stretch · Reconstruction error

Project `X` onto k = 1, 2, 3 components and back (`inverse_transform`). Store the mean squared reconstruction error for each k in the dict `err`. It must shrink as k grows and be ~0 at k = 3.

In [ ]:
err = {}   # TODO


In [ ]:
try:
    check("three entries", set(err) == {1, 2, 3})
    check("error decreases with k", err[1] > err[2] > err[3])
    check("no loss at full rank", err[3] < 1e-20)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
err = {}
for k in (1, 2, 3):
    p = PCA(n_components=k).fit(X)
    err[k] = float(np.mean((X - p.inverse_transform(p.transform(X))) ** 2))

```

PCA keeps the directions that minimise exactly this reconstruction error for a given k.

</details>

---
*Back to the course: **Machine Learning End To End → Principal Component Analysis**.*